In [0]:
from pyspark.sql.functions import *
from delta.tables import *

In [0]:
read_stream = (spark.readStream
                 .format("delta")
                 .table("silver.processed_btcusdt"))

In [0]:
aggregated_stream = (read_stream
                     .withWatermark("timestamp", "10 minutes")
                     .groupBy(
                         window(col('timestamp'), "10 minutes"),
                         col('side')
                     )
                     .agg(
                         sum("order_amount").alias('total_amount')
                         )
)

aggregated_stream = aggregated_stream.withColumns(
    {
    'start': col('window.start'),
    'end': col('window.end')}
    )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.btc_15min_metrics (
    start timestamp,
    end timestamp,
    side string,
    total_amount decimal(18, 3)
);

In [0]:
deltaTable = DeltaTable.forName(spark, "gold.btc_15min_metrics")

In [0]:
def upsertToDelta(microBatchOutputDF, batchId):
  (deltaTable.alias("t").merge(
      microBatchOutputDF.alias("s"),
      "s.start = t.start AND s.end = t.end AND s.side = t.side")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
  )


In [0]:
write_stream = (aggregated_stream.writeStream
    .format("delta")
    .outputMode("update")
    .foreachBatch(upsertToDelta)
    .trigger(availableNow=True)
    .option("checkpointLocation", "/Volumes/dbw_devfrancecentrallo1b/gold/gold_checkpoints") \
    .start()
)